In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [13]:
!pip install pytorch-crf

In [1]:
from datasets import load_from_disk
from datasets import concatenate_datasets
from datasets import DatasetDict
import os

In [2]:
ASTE_path = "/kaggle/input/edurabsa-aste/EduRABSA_ASTE"
WORK_PATH = "/kaggle/working/EduRABSA_ASTE"

dataset = load_from_disk(ASTE_path)
dataset.save_to_disk(WORK_PATH)

dataset = load_from_disk(WORK_PATH)
print(dataset)


Saving the dataset (0/1 shards):   0%|          | 0/4000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 4000
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 2500
    })
})


In [3]:
import os
import random
import numpy as np
from datasets import load_from_disk, DatasetDict, concatenate_datasets

# --- Cấu hình đường dẫn ---
ASTE_path = "/kaggle/input/edurabsa-aste/EduRABSA_ASTE"
WORK_PATH = "/kaggle/working/EduRABSA_ASTE_Split" # Lưu vào thư mục mới để tránh ghi đè lỗi

# --- Hàm hỗ trợ lấy mẫu theo chiến lược Paper ---
def paper_strategy_sample(dataset, n_samples, exclude_indices=set(), seed=42):
    """
    Hàm lấy mẫu mô phỏng chiến lược của paper:
    - Cố gắng cân bằng Course và Staff.
    - Tối đa hóa University (nếu có thông tin label).
    - Nếu không có thông tin label chi tiết, dùng random state cố định để đảm bảo phân phối ngẫu nhiên đều.
    """
    all_indices = set(range(len(dataset)))
    available_indices = list(all_indices - exclude_indices)
    
    # Nếu số lượng yêu cầu lớn hơn số lượng có sẵn
    if n_samples > len(available_indices):
        print(f"Cảnh báo: Yêu cầu {n_samples} nhưng chỉ còn {len(available_indices)} mẫu. Lấy tất cả.")
        n_samples = len(available_indices)
    
    # Ở đây chúng ta dùng Random State cố định để đảm bảo tái lập kết quả
    rng = np.random.RandomState(seed)
    selected_indices = rng.choice(available_indices, size=n_samples, replace=False)
    
    return set(selected_indices)

def create_splits(original_dataset):
    print("--- Bắt đầu chia dataset theo Paper ---")
    
    # 1. Xử lý tập TEST gốc (Nguồn cho Dev và Test mới)
    # Paper: "Development & Test sets are constructed from the default test set"
    src_test = original_dataset['test']
    print(f"Tổng số mẫu Test gốc: {len(src_test)}")
    
    # Tạo Development set (200 mẫu)
    dev_indices = paper_strategy_sample(src_test, n_samples=200, seed=42)
    dev_set = src_test.select(dev_indices)
    
    # Tạo Test set (300 mẫu) - Loại trừ các mẫu đã dùng cho Dev
    test_indices = paper_strategy_sample(src_test, n_samples=300, exclude_indices=dev_indices, seed=43)
    test_set = src_test.select(test_indices)
    
    print(f"-> Đã tạo Dev: {len(dev_set)} | Test: {len(test_set)}")

    # 2. Xử lý tập TRAIN gốc (Nguồn cho Validation và các tập Train mới)
    # Paper: "Train & Validation sets are extracted from the default training set"
    src_train = original_dataset['train']
    print(f"Tổng số mẫu Train gốc: {len(src_train)}")
    
    # Tạo In-training Validation set (200 mẫu)
    # Tập này phải tách biệt hoàn toàn để dùng cho Early Stopping
    val_indices = paper_strategy_sample(src_train, n_samples=200, seed=42)
    val_set = src_train.select(val_indices)
    
    # Tạo các tập Train với kích thước khác nhau (200, 500, 1000, 2000)
    # Lưu ý: Các tập train này lấy từ phần còn lại (loại trừ validation)
    train_subsets = {}
    remaining_indices_for_train = set(range(len(src_train))) - val_indices
    
    # List các kích thước train cần tạo theo paper
    target_sizes = [200, 500, 1000, 2000]
    
    for size in target_sizes:
        # Lấy mẫu từ phần còn lại. 
        # Paper không nói rõ các tập train con có lồng nhau hay không, 
        # nhưng thường ta lấy ngẫu nhiên độc lập từ vùng khả dụng.
        current_indices = paper_strategy_sample(
            src_train, 
            n_samples=size, 
            exclude_indices=val_indices, # Quan trọng: Không được trùng với Validation
            seed=100 + size # Seed khác nhau cho mỗi size
        )
        train_subsets[f'train_{size}'] = src_train.select(current_indices)
        print(f"-> Đã tạo Train_{size}: {len(train_subsets[f'train_{size}'])}")

    # 3. Tổng hợp lại thành DatasetDict
    final_splits = DatasetDict({
        'validation': val_set,
        'dev': dev_set,
        'test': test_set,
        # Chọn một tập train mặc định (ví dụ 2000) để lưu key 'train' chuẩn
        'train': train_subsets['train_2000'],
        # Lưu thêm các tập train nhỏ hơn nếu cần dùng riêng
        'train_200': train_subsets['train_200'],
        'train_500': train_subsets['train_500'],
        'train_1000': train_subsets['train_1000']
    })
    
    return final_splits

# --- Thực thi ---
# 1. Load dữ liệu
print(f"Loading from: {ASTE_path}")
dataset = load_from_disk(ASTE_path)

# 2. Thực hiện chia
split_dataset = create_splits(dataset)

# 3. Lưu kết quả
print(f"Saving to: {WORK_PATH}")
split_dataset.save_to_disk(WORK_PATH)

# 4. Kiểm tra lại
print("\n--- Kiểm tra kết quả sau khi lưu ---")
reloaded_dataset = load_from_disk(WORK_PATH)
print(reloaded_dataset)

# In thử 1 mẫu để xem cấu trúc
print("\nMẫu dữ liệu từ tập Test:")
print(reloaded_dataset['test'][0])

Loading from: /kaggle/input/edurabsa-aste/EduRABSA_ASTE
--- Bắt đầu chia dataset theo Paper ---
Tổng số mẫu Test gốc: 2500
-> Đã tạo Dev: 200 | Test: 300
Tổng số mẫu Train gốc: 4000
-> Đã tạo Train_200: 200
-> Đã tạo Train_500: 500
-> Đã tạo Train_1000: 1000
-> Đã tạo Train_2000: 2000
Saving to: /kaggle/working/EduRABSA_ASTE_Split


Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/300 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]


--- Kiểm tra kết quả sau khi lưu ---
DatasetDict({
    validation: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 200
    })
    dev: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 200
    })
    test: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 300
    })
    train: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 2000
    })
    train_200: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 200
    })
    train_500: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 500
    })
    train_1000: Dataset({
        features: ['id', 'task_type', 'original_id', 'text', 'output'],
        num_rows: 1000
    })
})

Mẫu dữ liệu từ tập Test:
{'id': 'ASTE_3025', 'task_type': 'ASTE', 'origin

In [4]:
from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
ds = load_dataset("yhua219/EduRABSA_ACD")

README.md: 0.00B [00:00, ?B/s]

train/data-00000-of-00001.arrow:   0%|          | 0.00/2.05M [00:00<?, ?B/s]

test/data-00000-of-00001.arrow:   0%|          | 0.00/1.32M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2500 [00:00<?, ? examples/s]

In [5]:
import numpy as np
from datasets import load_dataset, DatasetDict

# --- CẤU HÌNH ---
ASTE_DATASET_ID = "yhua219/EduRABSA_ASTE" # Bộ dữ liệu cần chia
REF_DATASET_ID = "yhua219/EduRABSA_ACD"   # Bộ tham chiếu để lấy Category
WORK_PATH = "/kaggle/working/EduRABSA_ASTE_Split"

# Seed cố định để đảm bảo tái lập
SEED_DEV = 42
SEED_TEST = 43
SEED_VAL = 42

# --- 1. TẢI DỮ LIỆU VÀ KIỂM TRA ĐỒNG BỘ ---
print(f"Đang tải Target: {ASTE_DATASET_ID} và Reference: {REF_DATASET_ID}...")
ds_aste = load_dataset(ASTE_DATASET_ID)
ds_ref = load_dataset(REF_DATASET_ID)

# Kiểm tra nhanh xem 2 bộ có khớp nhau không (tránh lệch dòng)
def check_alignment(ds1, ds2, split='train'):
    len1, len2 = len(ds1[split]), len(ds2[split])
    if len1 != len2:
        raise ValueError(f"Lệch số lượng mẫu! {split}: {len1} vs {len2}")
    # Kiểm tra thử mẫu đầu tiên xem text có giống nhau không
    # Lưu ý: Tên cột chứa text có thể khác nhau (ví dụ 'sentence' hoặc 'text')
    text_col1 = 'sentence' if 'sentence' in ds1[split].column_names else 'text'
    text_col2 = 'sentence' if 'sentence' in ds2[split].column_names else 'text'
    
    if ds1[split][0][text_col1] != ds2[split][0][text_col2]:
        print(f"⚠️ CẢNH BÁO: Text mẫu đầu tiên của {split} không khớp!")
        print(f"ASTE: {ds1[split][0][text_col1]}")
        print(f"REF:  {ds2[split][0][text_col2]}")
    else:
        print(f"✅ Đã khớp nối dữ liệu tập {split} ({len1} mẫu).")

check_alignment(ds_aste, ds_ref, 'train')
check_alignment(ds_aste, ds_ref, 'test')

# --- 2. HÀM XÁC ĐỊNH LOẠI REVIEW TỪ BỘ THAM CHIẾU (ACD) ---
def get_review_type_from_ref(example_ref):
    """
    Phân loại dựa trên bộ dữ liệu ACD (Reference).
    """
    # Tìm nhãn category trong bộ tham chiếu
    candidates = []
    if 'labels' in example_ref: candidates = example_ref['labels']
    elif 'pairs' in example_ref: candidates = [p.get('category', '') for p in example_ref['pairs']]
    elif 'category' in example_ref: candidates = [example_ref['category']]
    
    str_labels = str(candidates)
    if "University" in str_labels: return "UNIVERSITY"
    elif "Staff" in str_labels: return "STAFF"
    else: return "COURSE"

# --- 3. CHIẾN LƯỢC LẤY MẪU (Trả về danh sách Index) ---
def get_stratified_indices(dataset_ref, n_samples, exclude_indices=set(), seed=42):
    """
    Dùng bộ Reference để tính toán ra các INDEX cần lấy.
    """
    available_indices = list(set(range(len(dataset_ref))) - exclude_indices)
    rng = np.random.RandomState(seed)
    
    # Phân loại index dựa trên Reference Data
    buckets = {"UNIVERSITY": [], "STAFF": [], "COURSE": []}
    for idx in available_indices:
        rtype = get_review_type_from_ref(dataset_ref[idx])
        buckets[rtype].append(idx)
        
    for rtype in buckets: rng.shuffle(buckets[rtype])
        
    # Logic Paper: Max Uni -> Balance Staff/Course
    n_uni_avail = len(buckets["UNIVERSITY"])
    n_take_uni = min(n_samples, n_uni_avail)
    
    remaining = n_samples - n_take_uni
    n_take_staff = remaining // 2
    n_take_course = remaining - n_take_staff
    
    # Xử lý thiếu hụt
    n_staff_avail = len(buckets["STAFF"])
    if n_take_staff > n_staff_avail:
        diff = n_take_staff - n_staff_avail
        n_take_staff = n_staff_avail
        n_take_course += diff
    
    # Gom Index
    selected_indices = []
    selected_indices.extend(buckets["UNIVERSITY"][:n_take_uni])
    selected_indices.extend(buckets["STAFF"][:n_take_staff])
    selected_indices.extend(buckets["COURSE"][:n_take_course])
    
    return set(selected_indices)

# --- 4. THỰC HIỆN CHIA TRÊN BỘ ASTE ---
def create_aste_splits_using_ref(ds_target, ds_reference):
    print("\n--- BẮT ĐẦU CHIA TẬP ASTE DỰA TRÊN ACD ---")
    
    # A. Xử lý tập TEST
    print("Xử lý tập Test/Dev...")
    # Dùng REF để tính index
    dev_idxs = get_stratified_indices(ds_reference['test'], 200, seed=SEED_DEV)
    test_idxs = get_stratified_indices(ds_reference['test'], 300, exclude_indices=dev_idxs, seed=SEED_TEST)
    
    # Dùng index để cắt TARGET (ASTE)
    ds_dev = ds_target['test'].select(dev_idxs)
    ds_test = ds_target['test'].select(test_idxs)
    
    # B. Xử lý tập TRAIN
    print("Xử lý tập Train/Val...")
    val_idxs = get_stratified_indices(ds_reference['train'], 200, seed=SEED_VAL)
    ds_val = ds_target['train'].select(val_idxs)
    
    train_splits = {}
    sizes = [200, 500, 1000, 2000]
    for sz in sizes:
        print(f" -> Tạo Train_{sz}...")
        t_idxs = get_stratified_indices(ds_reference['train'], sz, exclude_indices=val_idxs, seed=100+sz)
        train_splits[f'train_{sz}'] = ds_target['train'].select(t_idxs)

    # C. Đóng gói
    final_dict = DatasetDict({
        'validation': ds_val,
        'dev': ds_dev,
        'test': ds_test,
        'train': train_splits['train_2000'],
        'train_200': train_splits['train_200'],
        'train_500': train_splits['train_500'],
        'train_1000': train_splits['train_1000']
    })
    
    return final_dict

# --- CHẠY ---
final_aste_dataset = create_aste_splits_using_ref(ds_aste, ds_ref)

print("\n--- KẾT QUẢ CUỐI CÙNG CHO ASTE ---")
print(final_aste_dataset)
print(f"Mẫu dữ liệu tập Train (ASTE): {final_aste_dataset['train'][0]}")

# Lưu
final_aste_dataset.save_to_disk(WORK_PATH)
print(f"Đã lưu tại {WORK_PATH}")

Đang tải Target: yhua219/EduRABSA_ASTE và Reference: yhua219/EduRABSA_ACD...


README.md: 0.00B [00:00, ?B/s]

train/data-00000-of-00001.arrow:   0%|          | 0.00/1.89M [00:00<?, ?B/s]

test/data-00000-of-00001.arrow:   0%|          | 0.00/1.22M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/4000 [00:00<?, ? examples/s]

...

Saving the dataset (0/1 shards):   0%|          | 0/2000 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/500 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1000 [00:00<?, ? examples/s]

Đã lưu tại /kaggle/working/EduRABSA_ASTE_Split


In [12]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter


In [7]:
def word_tokenize(text):
    return text.lower().split()


In [9]:
import os
from datasets import load_from_disk


# 2. Cấu hình đường dẫn
WORK_PATH = "/kaggle/working/EduRABSA_ASTE_Split"

# 3. Tải dataset
print(f"Đang tải dataset từ: {WORK_PATH}")
processed_dataset = load_from_disk(WORK_PATH)

# 4. Xây dựng từ điển (Vocab) dùng hàm của bạn
def build_vocab(dataset_split, min_freq=1):
    print("Đang xây dựng từ điển (Vocabulary)...")
    counter = Counter()
    
    for example in dataset_split:
        # Lấy văn bản
        text = example.get('sentence', example.get('text', ''))
        
        # Dùng hàm của bạn để tách từ và lowercase
        tokens = word_tokenize(text) 
        counter.update(tokens)

    # Thêm token đặc biệt
    # <PAD> = 0: Để đệm câu
    # <UNK> = 1: Để thay thế từ lạ
    vocab = {"<PAD>": 0, "<UNK>": 1}
    
    for word, freq in counter.items():
        if freq >= min_freq:
            vocab[word] = len(vocab)
            
    print(f"Hoàn tất! Kích thước từ điển: {len(vocab)}")
    return vocab

# 5. Xây dựng Tag Vocab (Cố định cho bài toán ASTE/BIO)
def build_tag_vocab():
    # Định nghĩa nhãn BIO chuẩn
    tags = ["O", "B-ASP", "I-ASP", "B-OPN", "I-OPN"]
    tag2idx = {"<PAD>": 0} # Pad tag thường gán ID 0
    
    for t in tags:
        tag2idx[t] = len(tag2idx)
    return tag2idx

# --- THỰC THI ---

# A. Tạo Word Vocab (Chỉ dùng tập TRAIN để tránh leak dữ liệu test)
vocab = build_vocab(processed_dataset["train"], min_freq=1)
VOCAB_SIZE = len(vocab)

# B. Tạo Tag Vocab
tag_vocab = build_tag_vocab()
NUM_TAGS = len(tag_vocab)

print("\n--- KẾT QUẢ ---")
print(f"Kích thước Word Vocab: {VOCAB_SIZE}")
print(f"Tag Vocab: {tag_vocab}")

# --- HÀM CHUYỂN ĐỔI QUAN TRỌNG KHI TRAIN ---
def text_to_indices(text, vocab):
    """
    Chuyển câu văn thành list các số index.
    BẮT BUỘC phải dùng word_tokenize giống lúc build vocab.
    """
    tokens = word_tokenize(text) # Sẽ lower() và split()
    return [vocab.get(token, vocab["<UNK>"]) for token in tokens]

# Test thử
sample_text = "The teacher is helpful"
sample_indices = text_to_indices(sample_text, vocab)

print(f"\nKiểm tra thử nghiệm:")
print(f"Input: '{sample_text}'")
print(f"Tokens: {word_tokenize(sample_text)}")
print(f"Indices: {sample_indices}")

Đang tải dataset từ: /kaggle/working/EduRABSA_ASTE_Split
Đang xây dựng từ điển (Vocabulary)...
Hoàn tất! Kích thước từ điển: 8907

--- KẾT QUẢ ---
Kích thước Word Vocab: 8907
Tag Vocab: {'<PAD>': 0, 'O': 1, 'B-ASP': 2, 'I-ASP': 3, 'B-OPN': 4, 'I-OPN': 5}

Kiểm tra thử nghiệm:
Input: 'The teacher is helpful'
Tokens: ['the', 'teacher', 'is', 'helpful']
Indices: [12, 1041, 154, 431]


In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import ast

LABELS = ["O", "B-ASP", "I-ASP", "B-OPI", "I-OPI"]
label2id = {l: i for i, l in enumerate(LABELS)}
id2label = {i: l for l, i in label2id.items()}

def create_bio_labels(tokens, triplets):
    labels = ["O"] * len(tokens)

    for t in triplets:
        # Xử lý linh hoạt: t có thể là dict hoặc list/tuple
        if isinstance(t, dict):
            aspect = t.get("aspect", "")
            opinion = t.get("opinion", "")
        elif isinstance(t, (list, tuple)) and len(t) >= 2:
            aspect = t[0]
            opinion = t[1]
        else:
            continue

        # 1. Gán nhãn Aspect
        if aspect and str(aspect).lower() != "null":
            asp_tokens = str(aspect).lower().split()
            for i in range(len(tokens) - len(asp_tokens) + 1):
                if tokens[i:i+len(asp_tokens)] == asp_tokens:
                    labels[i] = "B-ASP"
                    for j in range(1, len(asp_tokens)):
                        labels[i+j] = "I-ASP"

        # 2. Gán nhãn Opinion (Tránh ghi đè lên Aspect)
        if opinion and str(opinion).lower() != "null":
            op_tokens = str(opinion).lower().split()
            for i in range(len(tokens) - len(op_tokens) + 1):
                if tokens[i:i+len(op_tokens)] == op_tokens:
                    # CHECK QUAN TRỌNG: Chỉ gán nếu vị trí đó đang là 'O'
                    # (Để tránh Opinion đè mất Aspect nếu từ bị trùng)
                    if all(labels[k] == "O" for k in range(i, i+len(op_tokens))):
                        labels[i] = "B-OPI"
                        for j in range(1, len(op_tokens)):
                            labels[i+j] = "I-OPI"

    return labels

In [18]:
class AODataset(Dataset):
    def __init__(self, hf_dataset, vocab, tokenizer_func):
        self.data = hf_dataset
        self.vocab = vocab
        self.tokenizer = tokenizer_func # Hàm word_tokenize của bạn

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Lấy text (xử lý nhiều tên cột khác nhau)
        text = item.get("sentence", item.get("text", ""))
        
        # Lấy triplets (xử lý nếu nó đang ở dạng string)
        triplets = item.get("labels", item.get("output", []))
        if isinstance(triplets, str):
            try: triplets = ast.literal_eval(triplets)
            except: triplets = []

        # Tokenize
        tokens = self.tokenizer(text)
        
        # Tạo labels
        labels = create_bio_labels(tokens, triplets)

        # Convert to IDs
        input_ids = [self.vocab.get(t, self.vocab["<UNK>"]) for t in tokens]
        label_ids = [label2id[l] for l in labels]

        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "label_ids": torch.tensor(label_ids, dtype=torch.long),
            "len": len(input_ids) # Trả về độ dài thật để pack sequence sau này
        }

In [21]:
import torch
import torch.nn as nn
from torchcrf import CRF

class BiLSTM_CRF(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_labels, pad_idx=0):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)
        # batch_first=True để khớp với output của collate_fn
        self.lstm = nn.LSTM(embed_dim, hidden_dim // 2, bidirectional=True, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_labels)
        self.crf = CRF(num_labels, batch_first=True)
        self.pad_idx = pad_idx

    def forward(self, x, tags=None, mask=None):
        # 1. Tạo mask tự động nếu chưa có (True tại từ thật, False tại padding)
        if mask is None:
            mask = x != self.pad_idx
            
        # 2. Embedding & LSTM
        emb = self.embedding(x)
        lstm_out, _ = self.lstm(emb)
        emissions = self.fc(lstm_out)

        # 3. CRF Layer
        if tags is not None:
            # Tính Loss: mask=mask để bỏ qua padding
            # reduction='mean' để loss không quá to khi batch lớn
            log_likelihood = self.crf(emissions, tags, mask=mask, reduction='mean')
            return -log_likelihood  # Trả về Loss dương để minimize
        else:
            # Dự đoán (Decoding)
            return self.crf.decode(emissions, mask=mask)

In [19]:
def collate_fn(batch):
    # Tách dữ liệu từ batch list
    input_ids = [item["input_ids"] for item in batch]
    label_ids = [item["label_ids"] for item in batch]
    lengths = torch.tensor([item["len"] for item in batch])

    # Padding
    # padding_value=0 cho input (là <PAD> trong vocab)
    # padding_value=0 cho label (là 'O' trong label2id)
    padded_inputs = pad_sequence(input_ids, batch_first=True, padding_value=0)
    padded_labels = pad_sequence(label_ids, batch_first=True, padding_value=0)

    return padded_inputs, padded_labels, lengths

In [15]:
EPOCHS = 10
LR = 1e-3
BATCH_SIZE = 16   # nếu padding tốt


In [20]:
# Giả sử bạn đã có processed_dataset và vocab từ các bước trước

# 1. Khởi tạo Dataset
train_dataset = AODataset(processed_dataset["train"], vocab, word_tokenize)
test_dataset = AODataset(processed_dataset["test"], vocab, word_tokenize)

# 2. Khởi tạo DataLoader với collate_fn
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=True, 
    collate_fn=collate_fn # Đừng quên tham số này!
)

test_loader = DataLoader(
    test_dataset, 
    batch_size=BATCH_SIZE, 
    shuffle=False, 
    collate_fn=collate_fn
)

# 3. Test thử 1 batch
print("Checking batch shape...")
for inputs, labels, lens in train_loader:
    print(f"Input Shape: {inputs.shape}") # (32, Max_Len)
    print(f"Label Shape: {labels.shape}") # (32, Max_Len)
    print(f"Lengths: {lens}")
    break

Checking batch shape...
Input Shape: torch.Size([32, 68])
Label Shape: torch.Size([32, 68])
Lengths: tensor([15, 44, 36, 43, 27, 17, 68, 24,  4, 16, 33, 26, 20,  3, 16, 64,  8, 40,
        54, 37, 24, 26, 62, 29, 12, 33, 13, 15, 18,  1, 52, 60])


In [22]:
# Cấu hình thiết bị
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Training on: {device}")

# Tham số Hyperparameters
EMBED_DIM = 100
HIDDEN_DIM = 256
LEARNING_RATE = 0.001
EPOCHS = 10

# Khởi tạo Model
# vocab_size lấy từ biến VOCAB_SIZE ở các bước trước
# num_labels lấy từ len(label2id)
model = BiLSTM_CRF(vocab_size=VOCAB_SIZE, 
                   embed_dim=EMBED_DIM, 
                   hidden_dim=HIDDEN_DIM, 
                   num_labels=len(label2id),
                   pad_idx=0).to(device)

# Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

Training on: cpu


In [23]:
from tqdm import tqdm
import numpy as np

def train_epoch(model, dataloader, optimizer, device):
    model.train() # Chuyển sang chế độ train
    total_loss = 0
    
    # tqdm để hiện thanh tiến trình
    progress_bar = tqdm(dataloader, desc="Training")
    
    for batch in progress_bar:
        # 1. Lấy dữ liệu từ dataloader (nhớ khớp với collate_fn)
        # collate_fn trả về: input_ids, label_ids, lengths
        input_ids, label_ids, _ = batch 
        
        # 2. Chuyển lên GPU/CPU
        input_ids = input_ids.to(device)
        label_ids = label_ids.to(device)
        
        # 3. Tạo mask (token nào khác 0 là True)
        mask = (input_ids != 0)

        # 4. Zero grad
        optimizer.zero_grad()
        
        # 5. Forward & Loss
        loss = model(input_ids, tags=label_ids, mask=mask)
        
        # 6. Backward & Optimize
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})
        
    return total_loss / len(dataloader)

# --- CHẠY TRAINING ---
loss_history = []

print("--- BẮT ĐẦU HUẤN LUYỆN ---")
for epoch in range(EPOCHS):
    train_loss = train_epoch(model, train_loader, optimizer, device)
    loss_history.append(train_loss)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f}")

print("Đã huấn luyện xong!")

--- BẮT ĐẦU HUẤN LUYỆN ---


Training: 100%|██████████| 63/63 [00:20<00:00,  3.11it/s, loss=28.4]


Epoch 1/10 | Loss: 35.2582


Training: 100%|██████████| 63/63 [00:20<00:00,  3.12it/s, loss=19.3]


Epoch 2/10 | Loss: 26.3881


...

Epoch 8/10 | Loss: 12.0234


Training: 100%|██████████| 63/63 [00:15<00:00,  4.07it/s, loss=10.1]


Epoch 9/10 | Loss: 10.0843


Training: 100%|██████████| 63/63 [00:15<00:00,  3.97it/s, loss=9.17]

Epoch 10/10 | Loss: 8.4140
Đã huấn luyện xong!


In [24]:
from sklearn.metrics import classification_report

def evaluate(model, dataloader, device, id2label):
    model.eval()
    all_preds = []
    all_trues = []
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating"):
            input_ids, label_ids, _ = batch
            input_ids = input_ids.to(device)
            label_ids = label_ids.to(device)
            mask = (input_ids != 0)
            
            # Dự đoán (model trả về list các list nhãn ID)
            predictions = model(input_ids, mask=mask)
            
            # Chuyển đổi ID sang Text Label
            # Lưu ý: predictions không chứa padding, nhưng label_ids thì có
            # Ta cần lọc label_ids dựa trên mask để so sánh
            
            for i, pred_seq in enumerate(predictions):
                # Lấy nhãn thật tương ứng (loại bỏ padding dựa trên độ dài pred_seq)
                # Vì CRF decode tự động bỏ padding dựa vào mask
                true_seq = label_ids[i][:len(pred_seq)].cpu().numpy()
                
                # Convert ID -> Label String
                pred_labels = [id2label[p] for p in pred_seq]
                true_labels = [id2label[t] for t in true_seq]
                
                all_preds.extend(pred_labels)
                all_trues.extend(true_labels)
                
    # In báo cáo (dùng sklearn cho nhanh, hoặc seqeval nếu muốn strict BIO evaluation)
    print("\n--- Báo cáo kết quả (Token-level) ---")
    # Lọc bỏ nhãn 'O' nếu muốn xem kỹ hơn về ASP/OPI
    target_names = [l for l in list(id2label.values()) if l != 'O'] 
    print(classification_report(all_trues, all_preds, labels=target_names))

# --- CHẠY ĐÁNH GIÁ ---
evaluate(model, test_loader, device, id2label)

Evaluating: 100%|██████████| 10/10 [00:00<00:00, 15.90it/s]



--- Báo cáo kết quả (Token-level) ---
              precision    recall  f1-score   support

       B-ASP       0.58      0.53      0.55       542
       I-ASP       0.55      0.35      0.43       357
       B-OPI       0.57      0.48      0.52       625
       I-OPI       0.37      0.40      0.39      1289

   micro avg       0.47      0.44      0.45      2813
   macro avg       0.52      0.44      0.47      2813
weighted avg       0.48      0.44      0.45      2813



In [25]:
def predict_sentence(sentence, model, vocab, id2label, device):
    model.eval()
    # 1. Tokenize & Numericalize
    tokens = sentence.lower().split() # Dùng đúng hàm tokenizer lúc train
    input_ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    # 2. Predict
    with torch.no_grad():
        # Mask là toàn bộ True vì không có padding (batch size = 1)
        mask = torch.ones_like(input_tensor, dtype=torch.bool)
        prediction = model(input_tensor, mask=mask) # Trả về list of lists
        
    # 3. Decode
    pred_ids = prediction[0]
    pred_labels = [id2label[i] for i in pred_ids]
    
    # 4. In kết quả đẹp
    print(f"\nCâu: {sentence}")
    print(f"{'TOKEN':<15} {'PREDICTED TAG':<15}")
    print("-" * 30)
    for t, l in zip(tokens, pred_labels):
        print(f"{t:<15} {l:<15}")

# Thử
predict_sentence("The professors are very knowledgeable but the food is bad", model, vocab, id2label, device)


Câu: The professors are very knowledgeable but the food is bad
TOKEN           PREDICTED TAG  
------------------------------
the             O              
professors      B-ASP          
are             O              
very            B-OPI          
knowledgeable   I-OPI          
but             O              
the             O              
food            B-ASP          
is              O              
bad             B-OPI          


In [26]:
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Cấu hình chung
BATCH_SIZE = 32
EPOCHS = 10 # Số epoch cho mỗi lần thử nghiệm
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Danh sách các tập train cần chạy
# Lưu ý: 'train' trong processed_dataset chính là tập train_2000
subset_names = ['train_200', 'train_500', 'train_1000', 'train'] 

# Lưu kết quả để so sánh
experiment_results = {}

# Tập Test cố định (Dùng chung cho tất cả các lần train để so sánh công bằng)
test_dataset = AODataset(processed_dataset['test'], vocab, word_tokenize)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"--- BẮT ĐẦU THỰC NGHIỆM TRÊN {DEVICE} ---")

for name in subset_names:
    print(f"\n==========================================")
    print(f"🔄 Đang chạy với tập dữ liệu: {name.upper()}")
    print(f"==========================================")
    
    # 1. Tạo DataLoader cho tập con cụ thể (200, 500, 1000...)
    # Lấy đúng key từ processed_dataset
    current_train_data = processed_dataset[name] 
    
    train_ds = AODataset(current_train_data, vocab, word_tokenize)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    
    print(f"Số lượng mẫu huấn luyện: {len(train_ds)}")

    # 2. KHỞI TẠO LẠI MÔ HÌNH (Rất quan trọng!)
    # Phải tạo mới hoàn toàn để không dính trọng số cũ
    model = BiLSTM_CRF(
        vocab_size=len(vocab),
        embed_dim=100,
        hidden_dim=256,
        num_labels=len(label2id),
        pad_idx=0
    ).to(DEVICE)
    
    # Tạo lại Optimizer cho model mới
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    # 3. Training Loop
    best_loss = float('inf')
    
    for epoch in range(EPOCHS):
        # Gọi hàm train_epoch bạn đã viết ở bước trước
        train_loss = train_epoch(model, train_loader, optimizer, DEVICE)
        
        # In gọn mỗi epoch
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f"   Epoch {epoch+1}/{EPOCHS} | Loss: {train_loss:.4f}")
            
    # 4. Đánh giá sau khi train xong
    print(f"-> Đang đánh giá model {name} trên tập Test...")
    # Ở đây mình viết hàm eval đơn giản trả về loss hoặc accuracy/f1
    # Bạn có thể dùng hàm evaluate chi tiết ở bước trước để in report
    # Để đơn giản cho code này, ta chỉ chạy dự đoán để đảm bảo không lỗi
    
    # Gọi hàm evaluate để in classification report chi tiết ra màn hình
    print(f"--- Kết quả cho {name} ---")
    evaluate(model, test_loader, DEVICE, id2label)
    
    # Lưu lại model nếu cần
    torch.save(model.state_dict(), f"bilstm_crf_{name}.pth")
    print(f"✅ Đã xong {name}")

print("\n--- HOÀN TẤT TOÀN BỘ THỰC NGHIỆM ---")

--- BẮT ĐẦU THỰC NGHIỆM TRÊN cpu ---

🔄 Đang chạy với tập dữ liệu: TRAIN_200
Số lượng mẫu huấn luyện: 200


Training: 100%|██████████| 7/7 [00:01<00:00,  6.41it/s, loss=38.9]


   Epoch 1/10 | Loss: 48.9948


Training: 100%|██████████| 7/7 [00:01<00:00,  6.74it/s, loss=24]  


   Epoch 5/10 | Loss: 27.5226


...

   Epoch 5/10 | Loss: 18.6464


Training: 100%|██████████| 63/63 [00:15<00:00,  4.01it/s, loss=8.77]


   Epoch 10/10 | Loss: 8.9921
-> Đang đánh giá model train trên tập Test...
--- Kết quả cho train ---


Evaluating: 100%|██████████| 10/10 [00:00<00:00, 17.67it/s]



--- Báo cáo kết quả (Token-level) ---
              precision    recall  f1-score   support

       B-ASP       0.62      0.47      0.54       542
       I-ASP       0.61      0.24      0.35       357
       B-OPI       0.61      0.46      0.52       625
       I-OPI       0.54      0.31      0.40      1289

   micro avg       0.58      0.37      0.45      2813
   macro avg       0.60      0.37      0.45      2813
weighted avg       0.58      0.37      0.45      2813

✅ Đã xong train

--- HOÀN TẤT TOÀN BỘ THỰC NGHIỆM ---


In [ ]:
from tqdm import tqdm
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_ao = BiLSTM_CRF(VOCAB_SIZE, 100, 128, len(LABELS)).to(device)
optimizer = torch.optim.Adam(model_ao.parameters(), lr=1e-3)



train_loader = DataLoader(
    AODataset(processed_dataset["train"]),
    batch_size=1,
    shuffle=True
)

for epoch in range(EPOCHS):
    model_ao.train()
    total_loss = 0

    pbar = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        loss = model_ao(x, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # update progress bar
        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")


In [28]:
torch.save(model.state_dict(), "/kaggle/working/model_ao.pt")

In [29]:
def extract_spans(tokens, labels, span_type):
    spans = []
    i = 0
    while i < len(labels):
        if labels[i] == f"B-{span_type}":
            j = i + 1
            while j < len(labels) and labels[j] == f"I-{span_type}":
                j += 1
            spans.append(" ".join(tokens[i:j]))
            i = j
        else:
            i += 1
    return spans


In [32]:
def extract_spans(tokens, labels, span_type):
    spans = []
    i = 0
    while i < len(labels):
        if labels[i] == f"B-{span_type}":
            j = i + 1
            while j < len(labels) and labels[j] == f"I-{span_type}":
                j += 1
            spans.append(" ".join(tokens[i:j]))
            i = j
        else:
            i += 1
    return spans


In [33]:
SENTIMENT_MAP = {"Positive": 0, "Neutral": 1, "Negative": 2}

class SentimentDataset(Dataset):
    def __init__(self, hf_dataset):
        self.samples = []
        for row in hf_dataset:
            tokens = word_tokenize(row["text"])
            for t in row["output"]:
                self.samples.append((
                    tokens,
                    t["aspect"],
                    SENTIMENT_MAP[t["sentiment"]]
                ))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        tokens, aspect, label = self.samples[idx]
        ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
        return torch.tensor(ids), torch.tensor(label)


In [34]:
class BiLSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)
        self.attn = nn.Linear(hidden_dim * 2, 1)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)

    def forward(self, x):
        emb = self.embedding(x)
        h, _ = self.lstm(emb)
        weights = torch.softmax(self.attn(h).squeeze(-1), dim=1)
        rep = torch.sum(h * weights.unsqueeze(-1), dim=1)
        return self.fc(rep)


In [35]:
from torch.nn.utils.rnn import pad_sequence

PAD_ID = vocab["<PAD>"]

def sentiment_collate_fn(batch):
    """
    batch: list of (input_ids, label)
    """
    xs, ys = zip(*batch)

    xs_padded = pad_sequence(
        xs,
        batch_first=True,
        padding_value=PAD_ID
    )

    ys = torch.stack(ys)

    return xs_padded, ys


In [36]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model_sent = BiLSTM_Attention(VOCAB_SIZE, 100, 128, 3).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model_sent.parameters(), lr=1e-3)

loader = DataLoader(
    SentimentDataset(processed_dataset["train"]),
    batch_size=8,
    shuffle=True,
    collate_fn=sentiment_collate_fn
)

for epoch in range(EPOCHS):
    model_sent.train()
    total_loss = 0

    pbar = tqdm(
        loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}",
        leave=False
    )

    for x, y in pbar:
        x = x.to(device)
        y = y.to(device)

        logits = model_sent(x)
        loss = criterion(logits, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()



        total_loss += loss.item()

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}"
        })

    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1}/{EPOCHS} - Avg Loss: {avg_loss:.4f}")

Using device: cpu


Epoch 1/10 - Avg Loss: 0.8046


Epoch 2/10 - Avg Loss: 0.6474


Epoch 3/10 - Avg Loss: 0.5928


Epoch 4/10 - Avg Loss: 0.5640


...

Epoch 6/10 - Avg Loss: 0.5273


Epoch 7/10 - Avg Loss: 0.5176


Epoch 8/10 - Avg Loss: 0.5077


Epoch 9/10 - Avg Loss: 0.5016


Epoch 10/10 - Avg Loss: 0.4958


In [37]:
torch.save(model_sent.state_dict(), "/kaggle/working/model_sent_processed.pt")

In [38]:
def evaluate_ao(model, dataset, device):
    from collections import Counter

    tp = Counter()
    fp = Counter()
    fn = Counter()

    model.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])
            gold_triplets = sample["output"]

            gold_aspects = set(
                t["aspect"].lower()
                for t in gold_triplets
                if t["aspect"].lower() != "null"
            )
            gold_opinions = set(
                t["opinion"].lower()
                for t in gold_triplets
            )

            #  FIX: đưa ids lên đúng device
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            pred_aspects = set(extract_spans(tokens, pred_labels, "ASP"))
            pred_opinions = set(extract_spans(tokens, pred_labels, "OPI"))

            for a in pred_aspects:
                if a in gold_aspects:
                    tp["ASP"] += 1
                else:
                    fp["ASP"] += 1
            for a in gold_aspects - pred_aspects:
                fn["ASP"] += 1

            for o in pred_opinions:
                if o in gold_opinions:
                    tp["OPI"] += 1
                else:
                    fp["OPI"] += 1
            for o in gold_opinions - pred_opinions:
                fn["OPI"] += 1

    def prf(t, f_p, f_n):
        p = t / (t + f_p + 1e-8)
        r = t / (t + f_n + 1e-8)
        f1 = 2 * p * r / (p + r + 1e-8)
        return p, r, f1

    asp_scores = prf(tp["ASP"], fp["ASP"], fn["ASP"])
    opi_scores = prf(tp["OPI"], fp["OPI"], fn["OPI"])

    return asp_scores, opi_scores


In [39]:
def evaluate_ao_pair(model, dataset, device):
    """
    Đánh giá chính xác cặp (aspect, opinion)
    """

    tp, fp, fn = 0, 0, 0
    model.eval()

    with torch.no_grad():
        for sample in dataset:
            tokens = word_tokenize(sample["text"])

            # ===== GOLD AO PAIRS =====
            gold_pairs = set(
                (
                    t["aspect"].lower(),
                    t["opinion"].lower()
                )
                for t in sample["output"]
                if t["aspect"].lower() != "null"
            )

            # ===== MODEL PREDICTION =====
            ids = torch.tensor(
                [[vocab.get(t, 1) for t in tokens]],
                device=device
            )

            pred_label_ids = model(ids)[0]
            pred_labels = [id2label[i] for i in pred_label_ids]

            pred_aspects  = extract_spans(tokens, pred_labels, "ASP")
            pred_opinions = extract_spans(tokens, pred_labels, "OPI")

            # ===== PREDICTED AO PAIRS =====
            pred_pairs = set(
                (a.lower(), o.lower())
                for a in pred_aspects
                for o in pred_opinions
            )

            # ===== COUNT =====
            tp += len(pred_pairs & gold_pairs)
            fp += len(pred_pairs - gold_pairs)
            fn += len(gold_pairs - pred_pairs)

    precision = tp / (tp + fp + 1e-8)
    recall    = tp / (tp + fn + 1e-8)
    f1        = 2 * precision * recall / (precision + recall + 1e-8)

    return precision, recall, f1


In [40]:
p, r, f1 = evaluate_ao_pair(
    model,
    split_dataset["test"],
    device
)

print(f"AO Pair Precision: {p:.4f}")
print(f"AO Pair Recall   : {r:.4f}")
print(f"AO Pair F1       : {f1:.4f}")


NameError: name 'model_ao' is not defined

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

asp_scores, opi_scores = evaluate_ao(
    model_ao,
    processed_dataset["test"],
    device
)

print("Aspect Extraction (P/R/F1):", asp_scores)
print("Opinion Extraction (P/R/F1):", opi_scores)


In [41]:
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

def evaluate_sentiment(model, dataloader, device):
    model.eval()
    all_preds = []
    all_labels = []
    
    # Mapping ngược từ số về chữ (0 -> Positive)
    idx2sentiment = {v: k for k, v in SENTIMENT_MAP.items()}
    
    with torch.no_grad():
        for x, y in tqdm(dataloader, desc="Evaluating"):
            x = x.to(device)
            y = y.to(device)
            
            logits = model(x)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
            
    # Convert sang tên nhãn để in report cho dễ hiểu
    pred_names = [idx2sentiment[p] for p in all_preds]
    true_names = [idx2sentiment[l] for l in all_labels]
    
    print("\n--- Sentiment Classification Report ---")
    print(classification_report(true_names, pred_names))
    print(f"Accuracy: {accuracy_score(true_names, pred_names):.4f}")

# --- THỰC HIỆN ĐÁNH GIÁ ---
# Tạo loader cho tập Test
test_loader_sent = DataLoader(
    SentimentDataset(processed_dataset["test"]),
    batch_size=32,
    shuffle=False,
    collate_fn=sentiment_collate_fn
)

evaluate_sentiment(model_sent, test_loader_sent, device)

Evaluating: 100%|██████████| 38/38 [00:01<00:00, 31.64it/s]


--- Sentiment Classification Report ---
              precision    recall  f1-score   support

    Negative       0.62      0.61      0.61       387
     Neutral       0.26      0.05      0.09       128
    Positive       0.73      0.84      0.78       688

    accuracy                           0.68      1203
   macro avg       0.54      0.50      0.50      1203
weighted avg       0.64      0.68      0.65      1203

Accuracy: 0.6833


In [ ]:
import torch
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

def evaluate_sentiment_full(model, dataset, device, label_names=None):
    """
    model: BiLSTM_Attention
    dataset: SentimentDataset(processed_dataset["test"])
    device: cuda / cpu
    label_names: ["Positive", "Neutral", "Negative"]
    """

    model.eval()
    y_true = []
    y_pred = []

    loader = DataLoader(
        dataset,
        batch_size=16,
        shuffle=False,
        collate_fn=sentiment_collate_fn  # padding
    )

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            preds = torch.argmax(logits, dim=1)

            y_true.extend(y.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    # ===== Metrics =====
    acc = accuracy_score(y_true, y_pred)

    precision = precision_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    recall = recall_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    f1_macro = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )
    f1_weighted = f1_score(
        y_true, y_pred, average="weighted", zero_division=0
    )

    cm = confusion_matrix(y_true, y_pred)

    return {
        "Accuracy": acc,
        "Precision": precision,
        "Recall": recall,
        "Macro-F1": f1_macro,
        "Weighted-F1": f1_weighted,
        "Confusion Matrix": cm,
        "y_true": y_true,
        "y_pred": y_pred
    }


In [ ]:
label_names = ["Positive", "Neutral", "Negative"]

results = evaluate_sentiment_full(
    model_sent,
    SentimentDataset(processed_dataset["test"]),
    device,
    label_names
)


In [ ]:
print("===== Sentiment Classification Results =====")
print(f"Accuracy     : {results['Accuracy']:.4f}")
print(f"Precision    : {results['Precision']:.4f}")
print(f"Recall       : {results['Recall']:.4f}")
print(f"Macro-F1     : {results['Macro-F1']:.4f}")
print(f"Weighted-F1  : {results['Weighted-F1']:.4f}")

print("\nConfusion Matrix:")
print(results["Confusion Matrix"])


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def evaluate_sentiment(model, dataset):
    model.eval()
    y_true, y_pred = [], []

    with torch.no_grad():
        for x, y in DataLoader(dataset, batch_size=16):
            logits = model(x)
            preds = logits.argmax(dim=1)

            y_true.extend(y.tolist())
            y_pred.extend(preds.tolist())

    acc = accuracy_score(y_true, y_pred)
    f1  = f1_score(y_true, y_pred, average="macro")

    
    return acc, f1


In [ ]:
test_sent_ds = SentimentDataset(processed_dataset["test"])

acc, f1 = evaluate_sentiment(model_sent, test_sent_ds)

print(f"Sentiment Accuracy: {acc:.4f}")
print(f"Sentiment Macro-F1: {f1:.4f}")


In [43]:
def predict_absa(sentence, extract_model, sentiment_model, vocab, tag_map, device):
    """
    Hàm dự đoán trọn vẹn: Từ câu văn -> Danh sách (Aspect, Sentiment)
    """
    extract_model.eval()
    sentiment_model.eval()
    
    # 1. Xử lý đầu vào (Tokenize)
    tokens = sentence.lower().split() # Đảm bảo giống lúc train
    input_ids = [vocab.get(t, vocab["<UNK>"]) for t in tokens]
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)
    
    # 2. Bước 1: Trích xuất Aspect (Dùng model BiLSTM-CRF)
    # Giả sử extract_model là model CRF bạn đã train ở các bước trước
    with torch.no_grad():
        mask = (input_tensor != 0) # Mask đơn giản
        extract_preds = extract_model(input_tensor, mask=mask) # Trả về list of lists ID
        
    # Convert ID sang Tag (B-ASP, I-ASP...)
    # id2label là dictionary bạn đã tạo ở phần trước (0: 'O', 1: 'B-ASP'...)
    # Lưu ý: Cần đảm bảo bạn có biến id2label từ phần CRF
    pred_tags = [id2label[i] for i in extract_preds[0]]
    
    # Dùng hàm extract_spans bạn đã viết để lấy các cụm từ Aspect
    extracted_aspects = extract_spans(tokens, pred_tags, span_type="ASP")
    
    results = []
    
    # 3. Bước 2: Phân loại Sentiment cho từng Aspect
    if not extracted_aspects:
        return results # Không tìm thấy aspect nào
        
    # Sentiment Model hiện tại của bạn đang nhận đầu vào là CẢ CÂU VĂN
    # (Lưu ý: Model hiện tại chưa tối ưu vì nó không biết đang xét aspect nào, 
    # nhưng ta cứ chạy theo code hiện tại)
    
    with torch.no_grad():
        # Input cho sentiment model vẫn là câu gốc
        sent_logits = sentiment_model(input_tensor)
        sent_pred_id = torch.argmax(sent_logits, dim=1).item()
        
        # Map về chữ (Positive/Negative)
        idx2sentiment = {v: k for k, v in SENTIMENT_MAP.items()}
        sentiment_label = idx2sentiment[sent_pred_id]
        
        # Gán sentiment này cho TẤT CẢ aspect tìm được 
        # (Đây là hạn chế của kiến trúc hiện tại, xem phần Lưu ý bên dưới)
        for aspect in extracted_aspects:
            results.append((aspect, sentiment_label))
            
    return results

# --- CHẠY THỬ NGHIỆM ---
# Giả sử:
# - model_crf: Model BiLSTM-CRF đã train
# - model_sent: Model BiLSTM-Attention vừa train
# - vocab: Bộ từ điển dùng chung
# - id2label: Mapping tag của CRF

test_sentence = "The teachers are great but the food is terrible"

print(f"Câu: {test_sentence}")
print("Kết quả dự đoán:")
# Bạn cần truyền đúng model_crf đã train ở bước trước vào đây
predictions = predict_absa(test_sentence, model, model_sent, vocab, id2label, device)
print(predictions)

Câu: The teachers are great but the food is terrible
Kết quả dự đoán:
[('teachers', 'Negative'), ('food', 'Negative')]


In [45]:
test_sentence = "In depth stuff about King Arthur, the movie Excalibur doesn't seem that good anymore"

print(f"Câu: {test_sentence}")
print("Kết quả dự đoán:")
# Bạn cần truyền đúng model_crf đã train ở bước trước vào đây
predictions = predict_absa(test_sentence, model, model_sent, vocab, id2label, device)
print(predictions)

Câu: In depth stuff about King Arthur, the movie Excalibur doesn't seem that good anymore
Kết quả dự đoán:
[]


In [46]:
import pickle
import json

# 1. Lưu trọng số Model
torch.save(model.state_dict(), "model_crf.pth") # Model trích xuất
torch.save(model_sent.state_dict(), "model_sent.pth") # Model cảm xúc

# 2. Lưu Vocab & Config
# Vocab cần thiết để chuyển chữ thành số y hệt lúc train
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# Lưu các config cần thiết khác (như kích thước embed, hidden...)
config = {
    "embed_dim": 100,
    "hidden_dim": 256, # Hidden dim của CRF model
    "sent_hidden_dim": 128, # Hidden dim của Sentiment model
    "num_labels_crf": len(label2id),
    "num_classes_sent": 3
}
with open("config.json", "w") as f:
    json.dump(config, f)

print("Đã lưu xong các file cần thiết: .pth, .pkl, .json")

Đã lưu xong các file cần thiết: .pth, .pkl, .json
